# Exact MCMC, and checking you're calibrated

Two things any learned posterior needs: a way to be exact rather than approximately-right-probably, and a way to find out when it isn't.
```{note}
Tiny on purpose — a couple dozen modes, a couple thousand steps, a minute or two on a CPU. It's
here to show the API, not to be impressive. Real ones live in `FuncyFlows/examples/`.
```

In [ ]:
import matplotlib.pyplot as plt
import torch

from FuncyFlows.base_measures import CosineBasis, GaussianReferenceMeasure
from FuncyFlows.transports.continuous import (ContinuousTransformation, SumField,
                                              LinearField, MatrixField, TimeBasisConditioner)
from FuncyFlows.objectives import FlowMatching
from FuncyFlows.samplers import latent_pcn
from FuncyFlows.diagnostics import coverage_curve, coverage_error
from FuncyFlows.utils.gaussian_misfit import GaussianMisfit
from FuncyFlows.utils.train import train

torch.manual_seed(3)
DTYPE = torch.float64
M, NUM_OBS, NOISE = 20, 8, 0.15

basis = CosineBasis(M, dtype=DTYPE)
prior = GaussianReferenceMeasure(basis, alpha=0.05, power=2.0, dtype=DTYPE)
design_obs = basis.evaluate(torch.rand(NUM_OBS, 1, dtype=DTYPE))

## A learned prior plus a nasty likelihood

We observe `y = f(x)^2`. That's invariant under `f -> -f`, so the posterior is **bimodal** — there's no way to write it as one Gaussian blob and no amount of fitting will fix that. Good stress test for a sampler.

In [ ]:
bank = prior.sample(4000)
prior_flow = ContinuousTransformation(
    prior,
    SumField(LinearField(M, num_time_modes=4, dtype=DTYPE),
             MatrixField(TimeBasisConditioner(M, 96, num_time_modes=4, dtype=DTYPE),
                         mode_scale=prior.scale)),
    num_steps=12)
train(FlowMatching(prior_flow, bank, batch_size=128, weights=1 / prior.scale),
      prior_flow.parameters(), num_steps=1200, learning_rate=3e-3)

truth = prior.sample(1)
data = (truth @ design_obs.T)[0] ** 2 + NOISE * torch.randn(NUM_OBS, dtype=DTYPE)
misfit = GaussianMisfit(lambda v: (v @ design_obs.T) ** 2, data, NOISE)

## Latent pCN

pCN is [Cotter et al. (2013)](https://arxiv.org/abs/1202.0709); running it in a learned transport's latent space is [Parno & Marzouk (2018)](https://arxiv.org/abs/1412.5492).

The flow here is a learned **prior**, so the potential is the misfit on its own and the flow's Jacobian cancels out of the acceptance ratio entirely.

If the flow approximated the *posterior* instead you'd pass `misfit(v) + flow.log_rn_at(v)` — see the theory page. Getting this wrong doesn't raise anything, it just quietly samples the wrong distribution, so it's worth stopping to think about which one you've got.

In [ ]:
draws, info = latent_pcn(prior_flow, misfit, num_chains=64, num_steps=1500,
                         beta=0.4, thin=5, adapt_to=0.25)
print(f"acceptance {info['acceptance']:.2f} at final beta {info['beta']:.3f}")
print(f"mean potential over kept draws: {info['potential']:.2f}")

### Did it find both modes?

Project each draw onto the truth and look at the sign. A sampler stuck in one mode gives you ~0 or ~1. You want ~0.5.

In [ ]:
sign = ((draws @ truth[0]) > 0).double().mean()
print(f"fraction with positive overlap: {sign:.2f}   (0.50 = both modes)")

grid = torch.linspace(0, 1, 300, dtype=DTYPE)[:, None]
design = basis.evaluate(grid)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(grid[:, 0], (draws[:150] @ design.T).T, color="C0", alpha=0.06, lw=1)
ax.plot(grid[:, 0], (truth @ design.T)[0], "k", lw=1.5)
ax.plot(grid[:, 0], -(truth @ design.T)[0], "k--", lw=1.5)
ax.set_title("posterior draws; black = plus and minus the truth", fontsize=10)
plt.show()

```{warning}
If you pass `init`, watch `info["moved"]`. With a sharp likelihood the adapted `beta` can
collapse and the chains never actually leave where they started — and the output still looks
like a perfectly reasonable posterior, which is how this went unnoticed in our own inpainting
examples for a while. Under about 0.1 and you're looking at your initialisation.
```

## Coverage, via TARP

TARP is [Lemos et al. (2023)](https://arxiv.org/abs/2302.03026). One posterior tells you nothing about calibration; you need many cases. TARP takes each simulated truth, picks a random reference point, and counts how often the draws land closer to it than the truth does. Get it right and you get the diagonal.

We'll run it on an exactly-solvable setup so you can see a good curve and a deliberately broken one side by side.

In [ ]:
# a cheap, exactly-solvable setup so we can show a well-calibrated curve and a broken one
cases, draws_per_case = 200, 200
truths = prior.sample(cases)
observations = truths @ design_obs.T + NOISE * torch.randn(cases, NUM_OBS, dtype=DTYPE)

precision = torch.diag(1 / prior.variances) + design_obs.T @ design_obs / NOISE ** 2
cov = torch.linalg.inv(precision)
chol = torch.linalg.cholesky(0.5 * (cov + cov.T))
means = torch.linalg.solve(precision, (observations @ design_obs / NOISE ** 2).T).T

noise = torch.randn(cases, draws_per_case, M, dtype=DTYPE)
exact = means[:, None, :] + noise @ chol.T
narrow = means[:, None, :] + 0.6 * (noise @ chol.T)      # deliberately overconfident

In [ ]:
fig, ax = plt.subplots(figsize=(4.2, 4))
for name, samples in [("exact", exact), ("60% too narrow", narrow)]:
    levels, coverage = coverage_curve(samples, truths, weights=1 / prior.scale)
    ax.plot(levels, coverage, marker="o", ms=3, label=f"{name}  (error {coverage_error(levels, coverage):+.2f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8)
ax.set(xlabel="credibility level", ylabel="empirical coverage")
ax.legend(fontsize=8)
plt.show()

Below the diagonal means overconfident, above means too conservative. Both are wrong and they look nothing like each other, which is why the curve beats the single number.

---

That's the tour. `python -m FuncyFlows.examples` for the full-size versions.